In [ ]:
# Cell 0 — Imports & Configuration
import os
import re
import json
import time
import unicodedata
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# --- Paths ---
BASE_PATH       = Path("../../data/02-conferences/raw/")
OUTPUT_DIR      = Path("../../data/02-conferences/auxiliar/")
CHECKPOINT_PATH = OUTPUT_DIR / "v2_checkpoint.parquet"
FINAL_OUTPUT    = OUTPUT_DIR / "periodistas_v2_all_years.parquet"

# --- Config ---
YEAR_RANGE        = range(2018, 2025)  # change for testing, e.g. range(2021, 2022)
MODEL             = "gpt-4o-mini"
MAX_CHARS_PER_ROW = 150               # journalist names are always in the first sentence
RETRY_ATTEMPTS    = 3
RETRY_DELAY       = 5                 # seconds between retries
SAVE_INTERVAL     = 50                # save checkpoint every N files

# API key via environment variable — never hardcode
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
# Cell 1 — File Discovery
month_to_spanish = {
    1: "enero",    2: "febrero",   3: "marzo",     4: "abril",
    5: "mayo",     6: "junio",     7: "julio",     8: "agosto",
    9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}

def discover_all_files(base_path, year_range):
    """
    Walk the nested directory structure and return a sorted list of file records.
    Each record: {path, date, year, month, day}
    Only includes days where a PREGUNTA.CSV exists.
    """
    records = []
    for year in year_range:
        for month in range(1, 13):
            month_sp = month_to_spanish[month]
            month_dir = base_path / str(year) / f"{month}-{year}"
            if not month_dir.exists():
                continue
            for day in range(1, 32):
                day_dir = month_dir / f"{month_sp} {day}, {year}"
                for fname in ["PREGUNTA.CSV", "PREGUNTA.csv"]:
                    csv_path = day_dir / "csv_por_participante" / fname
                    if csv_path.exists():
                        try:
                            records.append({
                                "path":  csv_path,
                                "date":  pd.Timestamp(year, month, day).date(),
                                "year":  year,
                                "month": month,
                                "day":   day,
                            })
                        except ValueError:
                            pass  # invalid date (e.g. Feb 30)
                        break  # found the file, no need to try lowercase variant
    return sorted(records, key=lambda r: r["date"])

all_files = discover_all_files(BASE_PATH, YEAR_RANGE)
print(f"Discovered {len(all_files)} conference days")

In [ ]:
# Cell 2 — Prompt Design
SYSTEM_PROMPT = (
    "Eres un asistente especializado en extraer nombres de periodistas "
    "y sus medios de comunicación de transcripciones de conferencias de prensa del gobierno mexicano.\n\n"
    "Se te dará una lista numerada de fragmentos de texto. Cada fragmento es una línea de la "
    "transcripción. Los periodistas suelen presentarse usando frases como:\n"
    "- \"Soy [Nombre], de [Medio].\"\n"
    "- \"[Nombre], de [Medio].\"\n"
    "- \"Buenos días, [Nombre], de [Medio].\"\n"
    "- \"[Nombre], corresponsal de [Medio], de [ciudad].\"\n"
    "- \"Mi nombre es [Nombre], del periódico/canal/revista [Medio].\"\n"
    "- A veces la presentación está en dos líneas consecutivas: una con el saludo, otra con el nombre y medio.\n\n"
    "INSTRUCCIONES:\n"
    "1. Identifica todos los periodistas que se presentaron con su nombre.\n"
    "2. Extrae TODOS los medios que mencionaron (pueden ser varios).\n"
    "3. Si el periodista no mencionó ningún medio, usa \"outlets\": [] (lista vacía).\n"
    "4. Normaliza el nombre: capitalización correcta, sin títulos (señor, licenciado, etc.).\n"
    "5. No incluyas funcionarios del gobierno, al presidente, ni personas que hablen sin presentarse.\n"
    "6. Si no hay periodistas, devuelve [].\n\n"
    "FORMATO: JSON válido únicamente, sin texto adicional ni markdown.\n"
    "[{\"name\": \"Nombre Apellido\", \"outlets\": [\"Medio1\", \"Medio2\"]}, ...]"
)

def build_user_message(rows):
    """Format truncated rows as a numbered list."""
    return "\n".join(f"{i+1}. {text.strip()}" for i, text in enumerate(rows))

# Quick sanity check — print the prompt
print(SYSTEM_PROMPT)

In [ ]:
# Cell 3 — Extraction Functions

def read_pregunta_csv(path):
    """Try multiple encodings; return DataFrame or None."""
    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=encoding)
            if "Texto" in df.columns:
                return df
        except Exception:
            continue
    return None


def extract_journalists(file_record):
    """
    For one conference day:
    - Returns list of dicts: [{date, reporter, outlet, no_outlet_flag}, ...]
    - Returns []   if no journalists found (valid outcome)
    - Returns None if all retries failed (log for manual review)
    """
    df = read_pregunta_csv(file_record["path"])
    if df is None or df.empty:
        return []

    # Truncate each row — names appear in the first sentence
    rows = df["Texto"].fillna("").str[:MAX_CHARS_PER_ROW].tolist()
    user_msg = build_user_message(rows)

    for attempt in range(RETRY_ATTEMPTS):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0,
                response_format={"type": "json_object"},
                max_tokens=600,
                timeout=30,
            )
            raw = resp.choices[0].message.content.strip()
            parsed = json.loads(raw)

            # Model may return a list or wrap it in a dict key
            if isinstance(parsed, list):
                journalists = parsed
            elif isinstance(parsed, dict):
                journalists = (
                    parsed.get("journalists")
                    or parsed.get("periodistas")
                    or next((v for v in parsed.values() if isinstance(v, list)), [])
                )
            else:
                journalists = []

            results = []
            for j in journalists:
                name = j.get("name", "").strip()
                if not name:
                    continue
                outlets = j.get("outlets", [])
                if outlets:
                    for o in outlets:
                        o_clean = str(o).strip()
                        if o_clean:
                            results.append({
                                "date":           file_record["date"],
                                "reporter":       name,
                                "outlet":         o_clean,
                                "no_outlet_flag": False,
                            })
                else:
                    # Journalist detected but no outlet mentioned — flag for review
                    results.append({
                        "date":           file_record["date"],
                        "reporter":       name,
                        "outlet":         None,
                        "no_outlet_flag": True,
                    })
            return results

        except json.JSONDecodeError as e:
            print(f"  JSON error {file_record['date']} attempt {attempt+1}: {e}")
            time.sleep(RETRY_DELAY)
        except Exception as e:
            err = str(e).lower()
            wait = RETRY_DELAY * (2 ** attempt)  # exponential backoff for rate limits
            print(f"  Error {file_record['date']} attempt {attempt+1}: {e}")
            time.sleep(wait)

    print(f"  FAILED after {RETRY_ATTEMPTS} attempts: {file_record['date']}")
    return None

In [ ]:
# Cell 4 — Checkpoint (load prior progress if restarting)

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        df = pd.read_parquet(CHECKPOINT_PATH)
        done = set(df["date"].astype(str).unique())
        print(f"Checkpoint loaded: {len(done)} dates already processed")
        return df.to_dict("records"), done
    print("No checkpoint found — starting fresh")
    return [], set()

all_results, processed_dates = load_checkpoint()

In [ ]:
# Cell 5 — Main Processing Loop
failed_dates = []
remaining    = [f for f in all_files if str(f["date"]) not in processed_dates]
print(f"To process: {len(remaining)} | Already done: {len(processed_dates)}")

for i, file_record in enumerate(tqdm(remaining, desc="Extracting journalists")):
    result = extract_journalists(file_record)

    if result is None:
        failed_dates.append(str(file_record["date"]))
    else:
        all_results.extend(result)

    # Periodic checkpoint save
    if (i + 1) % SAVE_INTERVAL == 0:
        pd.DataFrame(all_results).to_parquet(CHECKPOINT_PATH, index=False)
        tqdm.write(f"  Checkpoint saved ({i+1} files done, {len(failed_dates)} failures)")

# Final checkpoint save
pd.DataFrame(all_results).to_parquet(CHECKPOINT_PATH, index=False)
print(f"\nDone. Total rows collected: {len(all_results)} | Failed dates: {len(failed_dates)}")

In [ ]:
# Cell 6 — Clean, Normalize & Save Final Output

def clean_text(text):
    """Lowercase, strip accents, remove non-alpha characters, collapse spaces."""
    if not text or pd.isna(text):
        return ""
    text = str(text).lower()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^a-z\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


df = pd.DataFrame(all_results)

df["reporter_aux"]   = df["reporter"].apply(clean_text)
df["outlet_aux"]     = df["outlet"].fillna("").apply(clean_text)
df["date"]           = pd.to_datetime(df["date"])
df["no_outlet_flag"] = df["no_outlet_flag"].astype(bool)

df = (
    df[["date", "reporter", "outlet", "no_outlet_flag", "reporter_aux", "outlet_aux"]]
    .sort_values(["date", "reporter_aux"])
    .reset_index(drop=True)
)

df.to_parquet(FINAL_OUTPUT, index=False)
print(f"Saved to {FINAL_OUTPUT}")
print(f"Shape: {df.shape}")
print(f"No-outlet flags: {df['no_outlet_flag'].sum()} rows to review manually")
df.head(10)

In [ ]:
# Cell 7 — Validation & Manual Review Export

# --- Summary stats ---
print("=== Summary ===")
print(f"Total rows:            {len(df)}")
print(f"Date range:            {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique dates:          {df['date'].nunique()}")
print(f"Unique reporters:      {df['reporter_aux'].nunique()}")
per_day = df.groupby("date").size()
print(f"Rows/day:              min={per_day.min()}  max={per_day.max()}  mean={per_day.mean():.1f}")

# --- Comparison with v1 (2021-2023 overlap) ---
v1_path = OUTPUT_DIR / "periodistas_long_2021_2023.parquet"
if v1_path.exists():
    df_v1   = pd.read_parquet(v1_path)
    overlap = df[df["date"].between("2021-01-01", "2023-12-31")]
    print("\n=== V1 vs V2 comparison (2021-2023) ===")
    print(f"V2 rows:              {len(overlap)}")
    print(f"V1 rows:              {len(df_v1)}")
    print(f"V2 unique reporters:  {overlap['reporter_aux'].nunique()}")
    print(f"V1 unique reporters:  {df_v1['reporter_aux'].nunique()}")

# --- Failed dates ---
if failed_dates:
    print(f"\nFailed dates ({len(failed_dates)}): {failed_dates[:5]}...")
    pd.Series(failed_dates, name="failed_date").to_csv(
        OUTPUT_DIR / "v2_failed_dates.csv", index=False
    )
    print(f"Saved failed dates to v2_failed_dates.csv")

# --- No-outlet cases for manual review ---
no_outlet = (
    df[df["no_outlet_flag"]]
    .drop_duplicates(["date", "reporter"])
    .sort_values(["date", "reporter_aux"])
    .reset_index(drop=True)
)
print(f"\nNo-outlet cases:       {len(no_outlet)} unique (reporter, date) pairs")
if len(no_outlet) > 0:
    review_path = OUTPUT_DIR / "v2_no_outlet_review.xlsx"
    no_outlet[["date", "reporter", "reporter_aux"]].to_excel(review_path, index=False)
    print(f"Saved for manual review → {review_path}")
    no_outlet[["date", "reporter"]].head(10)